### Import libraries

In [1]:
import numpy as np # type: ignore
import pandas as pd # type: ignore
import matplotlib.pyplot as plt # type: ignore
import plotly.express as px # type: ignore
import plotly.graph_objects as go # type: ignore
import plotly.io as pio # type: ignore
# import seaborn as sns # type: ignore
from array import *
from pickle import dump
from pathlib import Path
import json

# Ensure Plotly's renderer has a valid nbformat reference in this kernel session.
import nbformat # type: ignore
import plotly.io._renderers as _plotly_renderers # type: ignore
_plotly_renderers.nbformat = nbformat
pio.renderers.default = 'plotly_mimetype'

In [2]:
from sklearn.model_selection import train_test_split, KFold # type: ignore
from sklearn.metrics import mean_squared_error, mean_absolute_error # type: ignore
from sklearn.preprocessing import MinMaxScaler # type: ignore

In [3]:
import tensorflow as tf # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Dense, Flatten, LSTM, Conv1D, Dropout, MaxPooling1D, Conv1D, Input, Bidirectional, Add # type: ignore
from tensorflow.keras.layers import GRU, LayerNormalization, MultiHeadAttention, Embedding, Flatten, Attention, BatchNormalization, GlobalAveragePooling1D # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore
from tensorflow.keras.callbacks import EarlyStopping # type: ignore
from tensorflow.keras.models import Model # type: ignore

2026-04-08 07:51:02.202213: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-08 07:51:02.261300: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-08 07:51:03.699531: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Configure GPU usage

In [4]:
import tensorflow as tf # type: ignore
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
  tf.config.experimental.set_memory_growth(gpu, True)

W0000 00:00:1775634664.061896   82749 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


### Load and preprocess data

In [5]:
dataset = pd.read_csv('data/processed/YEGUAS_data.csv')
len(dataset)

3085

In [6]:
dataset.loc[:, dataset.columns != '_time'] = dataset.loc[:, dataset.columns != '_time'].astype('float32')
dataset['_time'] = pd.to_datetime(dataset['_time'])

In [7]:
dataset.set_index('_time', inplace=True)

In [8]:
dataset.columns

Index(['C4E_conductivity', 'C4E_salinity', 'C4E_tds', 'C4E_temperature',
       'NITRATE_nitrate_no3_mg_l', 'NTU_fnu', 'NTU_mgL', 'NTU_ntu',
       'NTU_temperature', 'OPTOD_mgL', 'OPTOD_ppm', 'OPTOD_saturation',
       'OPTOD_temperature'],
      dtype='object')

In [9]:
target = 'NITRATE_nitrate_no3_mg_l'
features = [col for col in dataset.columns if col != target]

In [10]:
# Put target at the end
dataset = dataset[features + [target]]

In [11]:
dataset.head()

,C4E_conductivity,C4E_salinity,C4E_tds,C4E_temperature,NTU_fnu,NTU_mgL,NTU_ntu,NTU_temperature,OPTOD_mgL,OPTOD_ppm,OPTOD_saturation,OPTOD_temperature,NITRATE_nitrate_no3_mg_l
_time,,,,,,,,,,,,,
2025-12-10 14:00:00,2254.0,1.21,1303.0,17.600000,6.460000,13.300000,6.460000,17.700001,7.89,7.89,80.599998,16.600000,53.400002
2025-12-10 14:30:00,2262.0,1.21,1308.0,17.500000,9.580000,17.799999,9.580000,17.700001,7.52,7.52,78.699997,17.700001,54.500000
2025-12-10 15:00:00,2286.0,1.23,1322.0,17.200001,15.600000,27.100000,15.600000,17.400000,7.51,7.51,78.199997,17.500000,54.099998
2025-12-10 15:30:00,2284.0,1.22,1320.0,17.299999,16.700001,28.600000,16.700001,17.500000,7.46,7.46,78.000000,17.700001,54.099998
2025-12-10 16:00:00,2289.0,1.23,1323.0,17.400000,18.600000,31.600000,18.600000,17.600000,7.38,7.38,77.699997,18.000000,53.900002


In [12]:
dataset.describe()

,C4E_conductivity,C4E_salinity,C4E_tds,C4E_temperature,NTU_fnu,NTU_mgL,NTU_ntu,NTU_temperature,OPTOD_mgL,OPTOD_ppm,OPTOD_saturation,OPTOD_temperature,NITRATE_nitrate_no3_mg_l
count,3085.000000,3085.000000,3085.000000,3085.000000,3085.000000,3085.000000,3085.000000,3085.000000,3085.000000,3085.000000,3085.000000,3085.000000,3085.000000
mean,2060.197245,1.104447,1147.839546,14.776305,17.568166,36.292777,17.568166,14.890723,7.217263,7.217034,71.552301,15.245993,55.350292
std,224.586990,0.120567,176.433329,1.704277,78.343852,124.795168,78.343852,1.731970,0.414438,0.414533,3.660422,1.755792,3.608265
min,1445.000000,0.764000,730.000000,9.360000,0.000000,0.000000,0.000000,9.380000,6.020000,6.020000,65.199997,9.890000,46.900002
25%,2041.000000,1.090000,1044.000000,13.600000,0.212000,3.850000,0.212000,13.700000,6.890000,6.890000,68.400002,13.900000,53.599998
50%,2119.000000,1.140000,1219.000000,14.800000,1.720000,6.120000,1.720000,15.000000,7.200000,7.200000,70.500000,15.400000,53.900002
75%,2207.000000,1.180000,1276.000000,15.900000,7.030000,14.100000,7.030000,16.100000,7.510000,7.510000,74.900002,16.600000,55.099998
max,2399.000000,1.290000,1409.000000,19.400000,1337.000000,1224.000000,1337.000000,19.500000,8.620000,8.620000,83.500000,20.299999,66.300003


In [13]:
dataset.columns

Index(['C4E_conductivity', 'C4E_salinity', 'C4E_tds', 'C4E_temperature',
       'NTU_fnu', 'NTU_mgL', 'NTU_ntu', 'NTU_temperature', 'OPTOD_mgL',
       'OPTOD_ppm', 'OPTOD_saturation', 'OPTOD_temperature',
       'NITRATE_nitrate_no3_mg_l'],
      dtype='object')

In [14]:
# Scale data
scaler = MinMaxScaler()
dataset[features] = scaler.fit_transform(dataset[features])

### Model architectures

In [15]:
def create_gru_model(n_inputs, n_outputs):
    model = Sequential()
    model.add(GRU(128, input_shape=(1, n_inputs), return_sequences=True))
    model.add(GRU(64))
    model.add(Dense(512))
    model.add(Dense(256))
    model.add(Dense(128))
    model.add(Dense(64))
    model.add(Dense(n_outputs))
    model.compile(optimizer='adam', loss='huber', metrics=['mae', 'mse', 'mape'])
    return model

### Training functions

In [16]:
def plot_overlay_predictions(y_true, y_pred, title="Actual vs Predicted"):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    x = np.arange(len(y_true))
 
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y_true, mode='lines', name='Actual', marker=dict(size=3)))
    fig.add_trace(go.Scatter(x=x, y=y_pred, mode='lines', name='Predicted',marker=dict(size=3)))
 
    fig.update_layout(
        title=title,
        xaxis_title="Time",
        yaxis_title="Value",
        template="plotly_white"
    )
    return fig

def plot_prediction_deltas(y_true, y_pred, title="Delta (Absolute Error)"):
    y_true = y_true.flatten()
    y_pred = y_pred.flatten()
    delta = np.abs(y_true - y_pred)
    x = np.arange(len(delta))
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=delta, mode='lines', name='Delta', marker=dict(size=3), line=dict(color='red')))
    
    fig.update_layout(
        title=title,
        xaxis_title="Time",
        yaxis_title="Absolute Error",
        template="plotly_white"
    )
    return fig

In [17]:
def create_sequences(dataset, back_seq_len):
    reference_time = 30 # min

    Xs, ys = [], []
    index_list = dataset.index
    for index in index_list:
        init_time = index - pd.Timedelta(minutes=back_seq_len*reference_time)
        print("Intendando: ", index, " con ventana: ", init_time, " - ", index)

        if init_time not in dataset.index:
            print("Ventana no válida para el índice: ", index)
            continue
        else:
            X_seq = dataset.loc[init_time:index, features].values
            y_seq = dataset.loc[index, target]
            if X_seq.shape[0] != back_seq_len + 1:
                print("Secuencia con longitud incorrecta para el índice: ", index)
                continue
            Xs.append(X_seq)
            ys.append(y_seq)
    return np.array(Xs), np.array(ys)

# def process_and_reshape(df):
#     x = np.array(df[features].values)
#     y = np.array(df[target].values)

#     num_samples = x.shape[0]
#     num_timesteps = 1
#     num_features = x.shape[1]

#     x = x.reshape(num_samples, num_timesteps, num_features)
#     y = y.reshape(num_samples, 1)

#     return x, y

### Training model

### Initializing MLflow
Configuración para el tracking del experimento con MLflow.

In [18]:
import mlflow
import mlflow.tensorflow

MLFLOW_EXPERIMENT = "nitrate_gru_experiment"
MLFLOW_TRACKING_URI = "http://mlflow.example.com:5000" # Puedes cambiarlo a tu IP como en el otro notebook si usas un servidor

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f'Experiment : {MLFLOW_EXPERIMENT}')
print(f'Tracking URI: {mlflow.get_tracking_uri()}')

/home/user/NITRATE_model/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Experiment : nitrate_gru_experiment
Tracking URI: http://mlflow.example.com:5000


In [19]:
df_train, df_val = train_test_split(dataset, test_size=0.1, shuffle=False)

In [20]:
EPOCHS = 50
BATCH_SIZE = 32

In [21]:
X_train, y_train = create_sequences(df_train, back_seq_len=8) # 5 horas
X_val, y_val = create_sequences(df_val, back_seq_len=8)

with mlflow.start_run(run_name='GRU_Baseline', tags={'architecture': 'GRU'}) as run:
    mlflow.tensorflow.autolog() # Autolog captura métricas por epoch de keras automáticamente
    
    # Registramos parámetros manuales también
    mlflow.log_params({
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "time_steps": X_train.shape[1],
        "n_features": X_train.shape[2]
    })
    
    # model = create_gru_model(X_train.shape[2], 1)
    # X_train.shape es típicamente (samples, time_steps, features)
    model = create_gru_model(X_train.shape[2], 1)
    
    history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(X_val, y_val), verbose=1, callbacks=[EarlyStopping(monitor='val_mae', mode='min', verbose=2, patience=25, restore_best_weights=True)])
    
    mae = history.history['mae'][-1]
    val_mae = history.history['val_mae'][-1]
    mse = history.history['mse'][-1] if 'mse' in history.history else None
    val_mse = history.history['val_mse'][-1]
    mape = history.history['mape'][-1]
    val_mape = history.history['val_mape'][-1]
    
    # Logeamos métricas finales manualmente como resumen general de MLflow
    metrics = {
        "final_mae": float(mae),
        "final_val_mae": float(val_mae),
        "final_val_mse": float(val_mse),
        "final_val_mape": float(val_mape)
    }
    if mse is not None:
        metrics["final_mse"] = float(mse)
    
    mlflow.log_metrics(metrics)
    
    # Generamos predicciones y gráficas para registrarlas como artefactos de MLflow
    y_pred = model.predict(X_val)
    pred_overlay = plot_overlay_predictions(y_val, y_pred)
    pred_delta = plot_prediction_deltas(y_val, y_pred)

    plot_dir = Path("outputs/artifacts/plots")
    plot_dir.mkdir(parents=True, exist_ok=True)

    overlay_path = plot_dir / "pred_overlay.html"
    delta_path = plot_dir / "pred_delta.html"

    pred_overlay.write_html(str(overlay_path), include_plotlyjs='cdn')
    pred_delta.write_html(str(delta_path), include_plotlyjs='cdn')

    mlflow.log_artifact(str(overlay_path), artifact_path='plots')
    mlflow.log_artifact(str(delta_path), artifact_path='plots')

    # También las guardamos vía MLflow como figuras (HTML).
    mlflow.log_figure(pred_overlay, "plots/pred_overlay_mlflow.html")
    mlflow.log_figure(pred_delta, "plots/pred_delta_mlflow.html")

    mlflow.log_param("mlflow_run_id", run.info.run_id)
    print(f"Run ID: {run.info.run_id}")
    print(f"MAE: {mae}, Val MAE: {val_mae}, MSE: {mse}, Val MSE: {val_mse}, MAPE: {mape}, Val MAPE: {val_mape}")

Intendando:  2025-12-10 14:00:00  con ventana:  2025-12-10 10:00:00  -  2025-12-10 14:00:00
Ventana no válida para el índice:  2025-12-10 14:00:00
Intendando:  2025-12-10 14:30:00  con ventana:  2025-12-10 10:30:00  -  2025-12-10 14:30:00
Ventana no válida para el índice:  2025-12-10 14:30:00
Intendando:  2025-12-10 15:00:00  con ventana:  2025-12-10 11:00:00  -  2025-12-10 15:00:00
Ventana no válida para el índice:  2025-12-10 15:00:00
Intendando:  2025-12-10 15:30:00  con ventana:  2025-12-10 11:30:00  -  2025-12-10 15:30:00
Ventana no válida para el índice:  2025-12-10 15:30:00
Intendando:  2025-12-10 16:00:00  con ventana:  2025-12-10 12:00:00  -  2025-12-10 16:00:00
Ventana no válida para el índice:  2025-12-10 16:00:00
Intendando:  2025-12-10 17:00:00  con ventana:  2025-12-10 13:00:00  -  2025-12-10 17:00:00
Ventana no válida para el índice:  2025-12-10 17:00:00
Intendando:  2025-12-10 17:30:00  con ventana:  2025-12-10 13:30:00  -  2025-12-10 17:30:00
Ventana no válida para el 

/home/user/NITRATE_model/.venv/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
78/79 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 19.7625 - mae: 20.2517 - mape: 37.2039 - mse: 768.1532

79/79 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - loss: 9.3340 - mae: 9.8096 - mape: 17.9257 - mse: 260.9390 - val_loss: 17.5824 - val_mae: 18.0824 - val_mape: 27.7867 - val_mse: 331.5745
Epoch 2/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 1.4752 - mae: 1.9211 - mape: 3.4687 - mse: 7.8120

79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - loss: 1.3415 - mae: 1.7839 - mape: 3.2381 - mse: 5.9699 - val_loss: 9.1643 - val_mae: 9.6643 - val_mape: 14.8489 - val_mse: 95.0716
Epoch 3/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.9148 - mae: 1.3422 - mape: 2.4429 - mse: 3.0358

79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 0.7309 - mae: 1.1296 - mape: 2.0598 - mse: 2.3542 - val_loss: 5.2464 - val_mae: 5.7464 - val_mape: 8.8266 - val_mse: 34.0117
Epoch 4/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.3827 - mae: 0.7280 - mape: 1.3373 - mse: 0.9980

... (3 outputs left out)

79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - loss: 0.2006 - mae: 0.4982 - mape: 0.9240 - mse: 0.5184 - val_loss: 0.3747 - val_mae: 0.7401 - val_mape: 1.1353 - val_mse: 0.8204
Epoch 10/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.2152 - mae: 0.5284 - mape: 0.9803 - mse: 0.5364 - val_loss: 1.4617 - val_mae: 1.9617 - val_mape: 3.0118 - val_mse: 4.0540
Epoch 11/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.2053 - mae: 0.5024 - mape: 0.9317 - mse: 0.5279 - val_loss: 0.6130 - val_mae: 1.0704 - val_mape: 1.6416 - val_mse: 1.3569
Epoch 12/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.1932 - mae: 0.4848 - mape: 0.9024 - mse: 0.5272

79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - loss: 0.2223 - mae: 0.5373 - mape: 0.9969 - mse: 0.5623 - val_loss: 0.1539 - val_mae: 0.4542 - val_mape: 0.7012 - val_mse: 0.3096
Epoch 13/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.1940 - mae: 0.4845 - mape: 0.8988 - mse: 0.5037 - val_loss: 0.8119 - val_mae: 1.2989 - val_mape: 1.9928 - val_mse: 1.8750
Epoch 14/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 0.2386 - mae: 0.5600 - mape: 1.0370 - mse: 0.5841 - val_loss: 0.3763 - val_mae: 0.7768 - val_mape: 1.1900 - val_mse: 0.7805
Epoch 15/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.2112 - mae: 0.5217 - mape: 0.9663 - mse: 0.5348 - val_loss: 0.6057 - val_mae: 1.0700 - val_mape: 1.6408 - val_mse: 1.3230
Epoch 16/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.2148 - mae: 0.5212 - mape: 0.9670 - mse: 0.5366 - val_loss: 0.7962 - val_mae: 1.2824 - val_mape: 1.9672 - val_mse: 1.8250
Epoch 17/50
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.1740 - mae: 0.4572 - mape: 0.8

2026/04/08 07:52:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/08 07:52:28 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during tensorflow autologging: BAD_REQUEST: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.IntegrityError) UNIQUE constraint failed: metrics.key, metrics.timestamp, metrics.step, metrics.run_uuid, metrics.value, metrics.is_nan
[SQL: INSERT INTO metrics ("key", value, timestamp, step, is_nan, run_uuid) VALUES (?, ?, ?, ?, ?, ?)]
[parameters: [('validation_mse', 331.5744934082031, 1775634673966, 0, 0, '9a3bea0a201f4b6e912c205f7f64d836'), ('validation_mse', 95.07159423828125, 1775634676164, 0, 0, '9a3bea0a201f4b6e912c205f7f64d836'), ('validation_mse', 34.0117073059082, 1775634678350, 0, 0, '9a3bea0a201f4b6e912c205f7f64d836'), ('validation_mse', 10.640236854553223, 1775634680522, 0, 0, '9a3bea0a201f4b6e91

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Run ID: 9a3bea0a201f4b6e912c205f7f64d836
MAE: 0.9286285638809204, Val MAE: 2.333820104598999, MSE: 1.2852368354797363, Val MSE: 5.848324775695801, MAPE: 1.7170443534851074, Val MAPE: 3.5806901454925537
🏃 View run GRU_Baseline at: http://mlflow.example.com:5000/#/experiments/4/runs/9a3bea0a201f4b6e912c205f7f64d836
🧪 View experiment at: http://mlflow.example.com:5000/#/experiments/4


In [22]:
model_name = 'prueba'

model_path = Path(f"test_models/{model_name}.h5")
scaler_path = Path(f"test_models/scaler_{model_name}.pkl")
features_path = Path(f"test_models/features_{model_name}.json")

model.save(str(model_path))
dump(scaler, open(scaler_path, 'wb'))
with open(features_path, 'w', encoding='utf-8') as f:
    json.dump(features, f, ensure_ascii=False, indent=2)

print(f"Model saved: {model_path}")
print(f"Scaler saved: {scaler_path}")
print(f"Features saved: {features_path}")

Model saved: test_models/prueba.h5
Scaler saved: test_models/scaler_prueba.pkl
Features saved: test_models/features_prueba.json


In [23]:
y_pred = model.predict(X_val)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


In [24]:
pred_overlay = plot_overlay_predictions(y_val, y_pred)
pred_overlay.show()

pred_delta = plot_prediction_deltas(y_val, y_pred)
pred_delta.show()